In [1]:
!pip uninstall -y torch torchvision torchaudio -q
!pip install -q torch --index-url https://download.pytorch.org/whl/cu130
!pip install -q "transformers==4.46.3" "accelerate>=1.0" "numpy<2" scipy huggingface_hub datasets tqdm safetensors sqlparse
print("✅ installed — RESTART KERNEL now, then run Cell 2")


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
✅ installed — RESTART KERNEL now, then run Cell 2


In [2]:
import os
from getpass import getpass
from huggingface_hub import login
os.environ["HF_HOME"] = "/workspace/.cache/huggingface"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
login(token=getpass("New HF token: "))
print("✅ logged in")

New HF token:  ········


✅ logged in


In [3]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="primal-sage/interpretability-workspace-backup", repo_type="model",
                  local_dir="/workspace", allow_patterns=["models/**", "data/**"])
snapshot_download(repo_id="primal-sage/tinysql-compression-results", repo_type="dataset",
                  local_dir="/workspace/gemma_results",
                  allow_patterns=["BASELINE_CORRECT.json", "ALL_SIGNALS_COMPLETE.npz",
                                  "tier_arrays.npz", "tacq_vulnerability.npz", "RETENTION_FINAL.json"])
os.makedirs("/workspace/reruns", exist_ok=True)
print("✅ downloads done")

Fetching 34 files:   0%|          | 0/34 [00:00<?, ?it/s]

test_CS1.json: 0.00B [00:00, ?B/s]

analysis_CS2.json: 0.00B [00:00, ?B/s]

analysis_CS1.json: 0.00B [00:00, ?B/s]

analysis_CS4.json: 0.00B [00:00, ?B/s]

...

tier_arrays.npz:   0%|          | 0.00/2.90k [00:00<?, ?B/s]

RETENTION_FINAL.json: 0.00B [00:00, ?B/s]

ALL_SIGNALS_COMPLETE.npz:   0%|          | 0.00/226M [00:00<?, ?B/s]

BASELINE_CORRECT.json: 0.00B [00:00, ?B/s]

✅ downloads done


In [4]:
import torch, transformers, os
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0), "| sm", torch.cuda.get_device_capability(0))
print("transformers", transformers.__version__)
for p in ["/workspace/models/finetuned/final/config.json", "/workspace/models/base/config.json",
          "/workspace/data/test_CS1.json", "/workspace/gemma_results/ALL_SIGNALS_COMPLETE.npz",
          "/workspace/gemma_results/BASELINE_CORRECT.json", "/workspace/gemma_results/tacq_vulnerability.npz"]:
    print("OK " if os.path.exists(p) else "MISSING ", p)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

torch 2.14.0+cu130 | cuda True | NVIDIA RTX PRO 4500 Blackwell | sm (12, 0)
transformers 4.46.3
OK  /workspace/models/finetuned/final/config.json
OK  /workspace/models/base/config.json
OK  /workspace/data/test_CS1.json
OK  /workspace/gemma_results/ALL_SIGNALS_COMPLETE.npz
OK  /workspace/gemma_results/BASELINE_CORRECT.json
OK  /workspace/gemma_results/tacq_vulnerability.npz


In [5]:
import torch, json, numpy as np, gc, time, os
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

DEVICE = torch.device('cuda:0')
N_LAYERS, N_HEADS, MLP_DIM = 26, 8, 9216
OUT = '/workspace/reruns'; os.makedirs(OUT, exist_ok=True)

model = AutoModelForCausalLM.from_pretrained('/workspace/models/finetuned/final',
                                             torch_dtype=torch.bfloat16, device_map='auto', local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained('/workspace/models/finetuned/final', local_files_only=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model.eval()

# data
with open('/workspace/gemma_results/BASELINE_CORRECT.json') as f: bc = json.load(f)
baseline_correct = bc['baseline_correct']
test_data_cache = {cs: json.load(open(f'/workspace/data/test_{cs}.json')) for cs in ['CS1','CS2','CS3','CS4','CS5']}

# signals + canonical tiers (identical to paper)
sigs = np.load('/workspace/gemma_results/ALL_SIGNALS_COMPLETE.npz')
def normalize(x):
    xmin, xmax = x.min(), x.max()
    return (x - xmin) / (xmax - xmin) if xmax > xmin else np.zeros_like(x)
norm_mlp = {'eap': normalize(sigs['eap_mlp']), 'gradient': normalize(sigs['grad_mlp']),
            'magnitude': normalize(sigs['mag_mlp']), 'weight_delta': normalize(sigs['wd_mlp']),
            'activation': normalize(sigs['act_delta_mlp']), 'edges': normalize(sigs['edge_mlp'])}
ta = np.load('/workspace/gemma_results/tier_arrays.npz')
mlp_tiers, attn_tiers = ta['mlp_tiers'], ta['attn_tiers']
tier_bits = {0:16, 1:8, 2:4, 3:0}
print('tier counts:', np.bincount(mlp_tiers.ravel(), minlength=4), '(expect 1295, 23, 238250, 48)')

# tier rule as a function (for the tau sweep)
def classify_tiers(tau=0.3, tau_supp=0.2, tau_low=0.1):
    t = np.full((N_LAYERS, MLP_DIM), 2, dtype=np.int8)
    S = np.stack([norm_mlp[k] for k in ['eap','gradient','magnitude','weight_delta','activation','edges']])
    high = (S >= tau).sum(0); low = (S < tau_low).sum(0)
    eap, grad, act = norm_mlp['eap'], norm_mlp['gradient'], norm_mlp['activation']
    skel = (high >= 3) | ((eap >= tau) & ((grad >= tau) | (act >= tau)))
    supp = (~skel) & (high >= 1) & ((eap >= tau_supp) | (grad >= tau_supp))
    prun = (~skel) & (~supp) & (low == 6)
    t[skel] = 0; t[supp] = 1; t[prun] = 3
    return t
print('rule check at tau=0.3 matches stored tiers:', np.array_equal(classify_tiers(0.3), mlp_tiers))

# quantizer (Cell J, per-group g=128)
def naive_quantize(tensor, bits, group_size=128):
    if bits >= 16: return tensor.clone()
    if bits == 0: return torch.zeros_like(tensor)
    t = tensor.float().reshape(-1)
    pad_len = ((t.numel() + group_size - 1) // group_size) * group_size
    padded = torch.zeros(pad_len, device=t.device); padded[:t.numel()] = t
    groups = padded.reshape(-1, group_size)
    gmin = groups.min(dim=1, keepdim=True).values; gmax = groups.max(dim=1, keepdim=True).values
    scale = ((gmax - gmin) / (2**bits - 1)).clamp(min=1e-10)
    q = torch.round((groups - gmin) / scale) * scale + gmin
    return q.reshape(-1)[:t.numel()].reshape(tensor.shape).to(tensor.dtype)

def apply_naive_tiered(mlp_tiers, mlp_tier_bits, attn_bits=8):
    for l in range(N_LAYERS):
        layer = model.model.layers[l]
        for proj_name in ['gate','up','down']:
            W = getattr(layer.mlp, f'{proj_name}_proj').weight.data
            for tier_val in range(4):
                bits = mlp_tier_bits[tier_val]
                idx = np.where(mlp_tiers[l] == tier_val)[0]
                if len(idx) == 0 or bits == 16: continue
                idx_t = torch.tensor(idx, device=W.device, dtype=torch.long)
                if proj_name in ['gate','up']: W[idx_t, :] = naive_quantize(W[idx_t, :], bits)
                else: W[:, idx_t] = naive_quantize(W[:, idx_t], bits)
        for proj in [layer.self_attn.q_proj, layer.self_attn.k_proj, layer.self_attn.v_proj, layer.self_attn.o_proj]:
            proj.weight.data = naive_quantize(proj.weight.data, attn_bits)

def apply_skeleton_quant(skeleton_mask, comp_bits=4, attn_bits=8):
    t = np.full((N_LAYERS, MLP_DIM), 2, dtype=np.int8); t[skeleton_mask] = 0
    apply_naive_tiered(t, {0:16, 1:comp_bits, 2:comp_bits, 3:comp_bits}, attn_bits)

# weight backup / restore (all 7 projections)
original_weights = {l: {n: getattr(model.model.layers[l].mlp if n in ['gate','up','down'] else model.model.layers[l].self_attn,
                        f'{n}_proj').weight.data.clone() for n in ['gate','up','down','q','k','v','o']} for l in range(N_LAYERS)}
def restore():
    for l in range(N_LAYERS):
        for n, W in original_weights[l].items():
            getattr(model.model.layers[l].mlp if n in ['gate','up','down'] else model.model.layers[l].self_attn,
                    f'{n}_proj').weight.data.copy_(W)

# evaluator (Cell D, verbatim)
def evaluate_retention():
    retained = 0; total = 0; per_cs = {}
    for cs in ['CS1','CS2','CS3','CS4','CS5']:
        idx_list = baseline_correct[cs]; total += len(idx_list); cs_ret = 0
        for i in idx_list:
            sample = test_data_cache[cs][i]
            inputs = tokenizer(sample['prompt'], return_tensors='pt', truncation=True, max_length=400).to(DEVICE)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.pad_token_id)
            gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
            if gen.split('\n')[0].strip() == sample['completion'].strip(): cs_ret += 1
        retained += cs_ret; per_cs[cs] = {'retained': cs_ret, 'total': len(idx_list)}
    return {'retention': round(100*retained/total, 2), 'retained': retained, 'total': total, 'per_cs': per_cs}

def run(name, apply_fn, **meta):
    restore(); apply_fn(); t0 = time.time()
    r = evaluate_retention(); r.update(meta); r['eval_s'] = round(time.time()-t0)
    restore()
    with open(f'{OUT}/{name}.json', 'w') as f: json.dump(r, f, indent=2)
    print(f"{name:32s} {r['retention']:6.2f}%  ({r['retained']}/{r['total']})  [{r['eval_s']}s]")
    return r

print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB — ready')

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

tier counts: [  1295     23 238250     48] (expect 1295, 23, 238250, 48)
rule check at tau=0.3 matches stored tiers: True
VRAM: 9.3 GB — ready


In [7]:
r0 = run('E0_repro_c4a8', lambda: apply_naive_tiered(mlp_tiers, tier_bits, 8), config='s16/s8/c4/p0 + attn@8')

E0_repro_c4a8                     96.19%  (101/105)  [469s]


In [8]:
from transformers import StoppingCriteria, StoppingCriteriaList
class StopOnNewline(StoppingCriteria):
    def __init__(self, start_len): self.start_len = start_len
    def __call__(self, input_ids, scores, **kw):
        return '\n' in tokenizer.decode(input_ids[0, self.start_len:], skip_special_tokens=True)

def evaluate_retention():
    retained = 0; total = 0; per_cs = {}
    for cs in ['CS1','CS2','CS3','CS4','CS5']:
        idx_list = baseline_correct[cs]; total += len(idx_list); cs_ret = 0
        for i in idx_list:
            sample = test_data_cache[cs][i]
            inputs = tokenizer(sample['prompt'], return_tensors='pt', truncation=True, max_length=400).to(DEVICE)
            L = inputs['input_ids'].shape[1]
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.pad_token_id,
                                     stopping_criteria=StoppingCriteriaList([StopOnNewline(L)]))
            gen = tokenizer.decode(out[0][L:], skip_special_tokens=True).strip()
            if gen.split('\n')[0].strip() == sample['completion'].strip(): cs_ret += 1
        retained += cs_ret; per_cs[cs] = {'retained': cs_ret, 'total': len(idx_list)}
    return {'retention': round(100*retained/total, 2), 'retained': retained, 'total': total, 'per_cs': per_cs}

# re-verify equivalence before trusting it
r0b = run('E0b_repro_c4a8_fastEval', lambda: apply_naive_tiered(mlp_tiers, tier_bits, 8), config='s16/s8/c4/p0 + attn@8')

E0b_repro_c4a8_fastEval           96.19%  (101/105)  [179s]


In [9]:
E1 = {}
for seed in [1, 2, 3, 4, 5]:
    np.random.seed(seed)
    m = np.zeros(N_LAYERS * MLP_DIM, dtype=bool)
    m[np.random.choice(N_LAYERS * MLP_DIM, 1295, replace=False)] = True
    mask = m.reshape(N_LAYERS, MLP_DIM)
    E1[seed] = run(f'E1_random_seed{seed}', lambda: apply_skeleton_quant(mask, 4, 8),
                   config='random 1295 @16, rest @4, attn @8', seed=seed)
vals = [E1[s]['retention'] for s in E1]
print(f"\nRandom 5-seed: mean={np.mean(vals):.1f}  std={np.std(vals, ddof=1):.1f}  min={min(vals):.1f}  max={max(vals):.1f}")
json.dump({'per_seed': vals, 'mean': np.mean(vals), 'std': np.std(vals, ddof=1)}, open(f'{OUT}/E1_random_summary.json','w'), indent=2)

E1_random_seed1                  100.00%  (105/105)  [172s]
E1_random_seed2                   99.05%  (104/105)  [173s]
E1_random_seed3                   98.10%  (103/105)  [175s]
E1_random_seed4                   97.14%  (102/105)  [175s]
E1_random_seed5                   99.05%  (104/105)  [173s]

Random 5-seed: mean=98.7  std=1.1  min=97.1  max=100.0


In [10]:
r2c = run('E2c_noskeleton_mlp4_a8', lambda: apply_skeleton_quant(np.zeros((N_LAYERS, MLP_DIM), dtype=bool), 4, 8),
          config='no skeleton, all MLP @4, attn @8')

E2c_noskeleton_mlp4_a8           100.00%  (105/105)  [175s]


In [11]:
def rand_mask(seed):
    np.random.seed(seed); m = np.zeros(N_LAYERS*MLP_DIM, dtype=bool)
    m[np.random.choice(N_LAYERS*MLP_DIM, 1295, replace=False)] = True
    return m.reshape(N_LAYERS, MLP_DIM)
skel_mask = (mlp_tiers == 0)
for tag, cb, ab in [('c3a8', 3, 8), ('c3a4', 3, 4)]:
    vals = [run(f'E1_{tag}_random_seed{s}', lambda: apply_skeleton_quant(rand_mask(s), cb, ab),
                config=f'random 1295 @16, rest @{cb}, attn @{ab}', seed=s)['retention'] for s in [1,2,3,4,5]]
    smart = run(f'E1_{tag}_tierskeleton', lambda: apply_skeleton_quant(skel_mask, cb, ab),
                config=f'tier-rule skeleton @16, rest @{cb}, attn @{ab}')['retention']
    print(f"\n{tag}: random mean={np.mean(vals):.1f} std={np.std(vals,ddof=1):.1f} min={min(vals):.1f} | smart={smart:.1f}\n")
    json.dump({'random_per_seed': vals, 'mean': np.mean(vals), 'std': np.std(vals,ddof=1), 'smart': smart},
              open(f'{OUT}/E1_{tag}_summary.json','w'), indent=2)

E1_c3a8_random_seed1              96.19%  (101/105)  [180s]
E1_c3a8_random_seed2              95.24%  (100/105)  [174s]
E1_c3a8_random_seed3              69.52%  (73/105)  [212s]
E1_c3a8_random_seed4              98.10%  (103/105)  [175s]
E1_c3a8_random_seed5              99.05%  (104/105)  [174s]
E1_c3a8_tierskeleton              97.14%  (102/105)  [174s]

c3a8: random mean=91.6 std=12.4 min=69.5 | smart=97.1

E1_c3a4_random_seed1              90.48%  (95/105)  [185s]
E1_c3a4_random_seed2              83.81%  (88/105)  [193s]
E1_c3a4_random_seed3              60.00%  (63/105)  [228s]
E1_c3a4_random_seed4              91.43%  (96/105)  [185s]
E1_c3a4_random_seed5              96.19%  (101/105)  [173s]
E1_c3a4_tierskeleton              94.29%  (99/105)  [182s]

c3a4: random mean=84.4 std=14.3 min=60.0 | smart=94.3



In [12]:
# E2d: no skeleton, MLP@3, attn@8  (matched-bits control for c3+a8)
run('E2d_noskeleton_mlp3_a8', lambda: apply_skeleton_quant(np.zeros((N_LAYERS, MLP_DIM), dtype=bool), 3, 8),
    config='no skeleton, all MLP @3, attn @8')

# E2a: 2-tier  (skeleton @16, everything else @4, attn @8)
run('E2a_2tier_c4a8', lambda: apply_naive_tiered(mlp_tiers, {0:16, 1:4, 2:4, 3:4}, 8), config='2-tier: s16, rest 4, attn 8')
# E2b: no supporting tier  (supporting -> 4-bit; prunable still 0)
run('E2b_nosupport_c4a8', lambda: apply_naive_tiered(mlp_tiers, {0:16, 1:4, 2:4, 3:0}, 8), config='s16/sup4/c4/p0, attn 8')
# E2e: no pruning  (prunable -> 4-bit; supporting still 8)  -- isolates whether the 48 zeroed neurons cost the 3pp
run('E2e_noprune_c4a8', lambda: apply_naive_tiered(mlp_tiers, {0:16, 1:8, 2:4, 3:4}, 8), config='s16/s8/c4/p4, attn 8')

# E3: tau sweep with full 4-tier rule, c4+a8
for tau in [0.25, 0.35]:
    t = classify_tiers(tau)
    counts = np.bincount(t.ravel(), minlength=4)
    run(f'E3_tau{tau}_c4a8', lambda: apply_naive_tiered(t, tier_bits, 8),
        config=f'tau={tau}, s16/s8/c4/p0, attn 8', tau=tau,
        skeleton=int(counts[0]), supporting=int(counts[1]), compressible=int(counts[2]), prunable=int(counts[3]))
    print(f"   tau={tau}: skeleton={counts[0]} ({100*counts[0]/(N_LAYERS*MLP_DIM):.2f}%) supp={counts[1]} prune={counts[3]}")

E2d_noskeleton_mlp3_a8            98.10%  (103/105)  [181s]
E2a_2tier_c4a8                   100.00%  (105/105)  [176s]
E2b_nosupport_c4a8                97.14%  (102/105)  [183s]
E2e_noprune_c4a8                 100.00%  (105/105)  [176s]
E3_tau0.25_c4a8                   95.24%  (100/105)  [182s]
   tau=0.25: skeleton=4074 (1.70%) supp=37 prune=48
E3_tau0.35_c4a8                   96.19%  (101/105)  [182s]
   tau=0.35: skeleton=339 (0.14%) supp=23 prune=48


In [13]:
def apply_protect(mask, comp_bits, attn_bits):
    """Quantize full matrices (aligned g=128 groups), then restore protected neurons from original."""
    for l in range(N_LAYERS):
        layer = model.model.layers[l]
        keep = torch.tensor(np.where(mask[l])[0], device=DEVICE, dtype=torch.long)
        for name in ['gate','up','down']:
            proj = getattr(layer.mlp, f'{name}_proj')
            proj.weight.data = naive_quantize(original_weights[l][name], comp_bits)
            if len(keep):
                if name in ['gate','up']: proj.weight.data[keep, :] = original_weights[l][name][keep, :]
                else:                     proj.weight.data[:, keep] = original_weights[l][name][:, keep]
        for name in ['q','k','v','o']:
            getattr(layer.self_attn, f'{name}_proj').weight.data = naive_quantize(original_weights[l][name], attn_bits)

none_ = run('E1fix_c3a8_none',    lambda: apply_protect(np.zeros((N_LAYERS, MLP_DIM), bool), 3, 8), config='protect-restore, none')
skel_ = run('E1fix_c3a8_skeleton', lambda: apply_protect(mlp_tiers == 0, 3, 8), config='protect-restore, tier skeleton')
vals = [run(f'E1fix_c3a8_random{s}', lambda: apply_protect(rand_mask(s), 3, 8), config='protect-restore, random', seed=s)['retention']
        for s in [1,2,3,4,5]]
print(f"\nFIXED c3a8 — none={none_['retention']:.1f}  skeleton={skel_['retention']:.1f}  "
      f"random={np.mean(vals):.1f}±{np.std(vals,ddof=1):.1f} (min {min(vals):.1f})")
json.dump({'none': none_['retention'], 'skeleton': skel_['retention'], 'random': vals},
          open(f'{OUT}/E1fix_c3a8_summary.json','w'), indent=2)

E1fix_c3a8_none                   98.10%  (103/105)  [166s]
E1fix_c3a8_skeleton              100.00%  (105/105)  [166s]
E1fix_c3a8_random1                99.05%  (104/105)  [168s]
E1fix_c3a8_random2                99.05%  (104/105)  [173s]
E1fix_c3a8_random3                98.10%  (103/105)  [173s]
E1fix_c3a8_random4                99.05%  (104/105)  [175s]
E1fix_c3a8_random5                98.10%  (103/105)  [175s]

FIXED c3a8 — none=98.1  skeleton=100.0  random=98.7±0.5 (min 98.1)


In [14]:
def apply_neuron_bits(bits_arr, attn_bits=8):
    """bits_arr: (26, 9216) int in {16,8,4,3,0}. Full-matrix aligned quantization per bit-width, composed by mask."""
    for l in range(N_LAYERS):
        layer = model.model.layers[l]; b = bits_arr[l]
        for name in ['gate','up','down']:
            W0 = original_weights[l][name]; Wq = W0.clone()
            for bits in [8, 4, 3, 2, 0]:
                idx = torch.tensor(np.where(b == bits)[0], device=DEVICE, dtype=torch.long)
                if len(idx) == 0: continue
                Qb = naive_quantize(W0, bits)
                if name in ['gate','up']: Wq[idx, :] = Qb[idx, :]
                else:                     Wq[:, idx] = Qb[:, idx]
            getattr(layer.mlp, f'{name}_proj').weight.data = Wq
        for name in ['q','k','v','o']:
            getattr(layer.self_attn, f'{name}_proj').weight.data = naive_quantize(original_weights[l][name], attn_bits)

def tier_bits_arr(tiers, bmap): 
    return np.vectorize(bmap.get)(tiers).astype(int)

# VulnSplit masks (Cell H logic: least-vulnerable fraction of compressible -> 3-bit)
tv = np.load('/workspace/gemma_results/tacq_vulnerability.npz', allow_pickle=True)
print('tacq keys:', {k: tv[k].shape for k in tv.keys()})
vkey = [k for k in tv.keys() if tv[k].shape == (N_LAYERS, MLP_DIM) and '3' in k]
vkey = vkey[0] if vkey else [k for k in tv.keys() if tv[k].shape == (N_LAYERS, MLP_DIM)][0]
vuln = tv[vkey]; print('using', vkey)
comp = (mlp_tiers == 2); pos = vuln[comp & (vuln > 0)]
def vulnsplit_bits(frac):
    thr = np.percentile(pos, frac*100)
    b = tier_bits_arr(mlp_tiers, tier_bits)          # s16 / s8 / c4 / p0
    b[comp & (vuln <= thr)] = 3                        # least-vulnerable frac of compressible -> 3-bit
    print(f'   vulnsplit {int(frac*100)}%: {(b==3).sum()} @3b, {(b==4).sum()} @4b')
    return b

T1 = {}
T1['c4a8']  = run('T1_aligned_c4a8',   lambda: apply_neuron_bits(tier_bits_arr(mlp_tiers, tier_bits), 8), config='aligned s16/s8/c4/p0 a8')
T1['c3a8']  = run('T1_aligned_c3a8',   lambda: apply_neuron_bits(tier_bits_arr(mlp_tiers, {0:16,1:8,2:3,3:0}), 8), config='aligned s16/s8/c3/p0 a8')
T1['v50']   = run('T1_aligned_vuln50', lambda: apply_neuron_bits(vulnsplit_bits(0.50), 8), config='aligned VulnSplit 50%@3b a8')
T1['v75']   = run('T1_aligned_vuln75', lambda: apply_neuron_bits(vulnsplit_bits(0.75), 8), config='aligned VulnSplit 75%@3b a8')
T1['2tier_c3a8'] = run('T1_aligned_2tier_c3a8', lambda: apply_neuron_bits(tier_bits_arr(mlp_tiers, {0:16,1:3,2:3,3:3}), 8), config='aligned s16 rest3 a8')

tacq keys: {'vuln_2bit': (26, 9216), 'vuln_3bit': (26, 9216)}
using vuln_3bit
T1_aligned_c4a8                   91.43%  (96/105)  [194s]
T1_aligned_c3a8                   99.05%  (104/105)  [179s]
   vulnsplit 50%: 119125 @3b, 119125 @4b
T1_aligned_vuln50                 95.24%  (100/105)  [188s]
   vulnsplit 75%: 178687 @3b, 59563 @4b
T1_aligned_vuln75                 95.24%  (100/105)  [185s]
T1_aligned_2tier_c3a8            100.00%  (105/105)  [178s]


In [15]:
run('T1_aligned_c4a8_noprune',  lambda: apply_neuron_bits(tier_bits_arr(mlp_tiers, {0:16,1:8,2:4,3:4}), 8), config='aligned s16/s8/c4/p4 a8')
run('T1_aligned_2tier_c4a8',    lambda: apply_neuron_bits(tier_bits_arr(mlp_tiers, {0:16,1:4,2:4,3:4}), 8), config='aligned s16 rest4 a8')
run('T1_aligned_none_c4a8',     lambda: apply_neuron_bits(np.full((N_LAYERS, MLP_DIM), 4), 8), config='aligned all MLP4 a8')
run('T1_aligned_c4a8_repeat',   lambda: apply_neuron_bits(tier_bits_arr(mlp_tiers, tier_bits), 8), config='aligned s16/s8/c4/p0 a8 (repeat)')

T1_aligned_c4a8_noprune           99.05%  (104/105)  [179s]
T1_aligned_2tier_c4a8             99.05%  (104/105)  [181s]
T1_aligned_none_c4a8             100.00%  (105/105)  [175s]
T1_aligned_c4a8_repeat            91.43%  (96/105)  [191s]


{'retention': 91.43,
 'retained': 96,
 'total': 105,
 'per_cs': {'CS1': {'retained': 22, 'total': 28},
  'CS2': {'retained': 25, 'total': 25},
  'CS3': {'retained': 22, 'total': 23},
  'CS4': {'retained': 14, 'total': 15},
  'CS5': {'retained': 13, 'total': 14}},
 'config': 'aligned s16/s8/c4/p0 a8 (repeat)',
 'eval_s': 191}

In [16]:
from datasets import load_dataset
wt = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
enc = tokenizer('\n\n'.join(wt['text']), return_tensors='pt').input_ids
def wikitext_ppl(seqlen=2048):
    nlls, n = [], 0
    for i in range(0, enc.shape[1] - 1, seqlen):
        ids = enc[:, i:i+seqlen].to(DEVICE)
        if ids.shape[1] < 2: break
        with torch.no_grad(): loss = model(ids, labels=ids).loss.float()
        nlls.append(loss * (ids.shape[1]-1)); n += ids.shape[1]-1
    return round(torch.exp(torch.stack(nlls).sum() / n).item(), 2)

ppl = {}
def ppl_run(name, apply_fn):
    restore(); apply_fn(); ppl[name] = wikitext_ppl(); restore(); print(f'{name:28s} PPL={ppl[name]}')
ppl_run('fp16', lambda: None)
ppl_run('uniform_4bit', lambda: apply_neuron_bits(np.full((N_LAYERS, MLP_DIM), 4), 4))
ppl_run('none_mlp4_a8', lambda: apply_neuron_bits(np.full((N_LAYERS, MLP_DIM), 4), 8))
ppl_run('2tier_c4a8', lambda: apply_neuron_bits(tier_bits_arr(mlp_tiers, {0:16,1:4,2:4,3:4}), 8))
ppl_run('none_mlp3_a8', lambda: apply_neuron_bits(np.full((N_LAYERS, MLP_DIM), 3), 8))
ppl_run('2tier_c3a8', lambda: apply_neuron_bits(tier_bits_arr(mlp_tiers, {0:16,1:3,2:3,3:3}), 8))
json.dump(ppl, open(f'{OUT}/E4_wikitext_ppl.json','w'), indent=2)

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

fp16                         PPL=6466.33
uniform_4bit                 PPL=10558.74
none_mlp4_a8                 PPL=10171.08
2tier_c4a8                   PPL=10400.93
none_mlp3_a8                 PPL=8101.61
2tier_c3a8                   PPL=8781.19


In [17]:
# E5: single-signal TaCQ-style selection — top-1295 most vulnerable neurons @16, rest @3, attn 8
flat = vuln.ravel(); tacq_mask = np.zeros_like(flat, dtype=bool)
tacq_mask[np.argpartition(flat, -1295)[-1295:]] = True; tacq_mask = tacq_mask.reshape(N_LAYERS, MLP_DIM)
run('E5_tacq_top1295_c3a8', lambda: apply_protect(tacq_mask, 3, 8), config='TaCQ vuln top-1295 @16, rest @3, attn 8')

# E7: percentile skeleton (same >=3-signal rule, per-signal P-th percentile thresholds)
S = np.stack([norm_mlp[k] for k in ['eap','gradient','magnitude','weight_delta','activation','edges']])
def percentile_skeleton(P):
    thr = np.percentile(S.reshape(6, -1), P, axis=1)[:, None, None]
    high = (S >= thr).sum(0)
    eap, grad, act = (S[0] >= thr[0]), (S[1] >= thr[1]), (S[4] >= thr[4])
    return (high >= 3) | (eap & (grad | act))
for P in [99, 99.5]:
    m = percentile_skeleton(P); k = int(m.sum())
    print(f'P={P}: skeleton={k} ({100*k/(N_LAYERS*MLP_DIM):.2f}%), overlap with tau=0.3 skeleton = {int((m & (mlp_tiers==0)).sum())}/1295')
    run(f'E7_P{P}_c3a8', lambda: apply_protect(m, 3, 8), config=f'percentile P={P} skeleton @16, rest @3, attn 8', P=P, skeleton=k)

E5_tacq_top1295_c3a8              98.10%  (103/105)  [177s]
P=99: skeleton=137 (0.06%), overlap with tau=0.3 skeleton = 32/1295
E7_P99_c3a8                       99.05%  (104/105)  [181s]
P=99.5: skeleton=51 (0.02%), overlap with tau=0.3 skeleton = 20/1295
E7_P99.5_c3a8                     99.05%  (104/105)  [179s]


In [18]:
bos = torch.tensor([[tokenizer.bos_token_id]])
body = enc[:, 1:] if enc[0,0].item() == tokenizer.bos_token_id else enc
def wikitext_ppl(seqlen=2048):
    nlls, n = [], 0
    for i in range(0, body.shape[1], seqlen - 1):
        chunk = body[:, i:i+seqlen-1]
        if chunk.shape[1] < 2: break
        ids = torch.cat([bos, chunk], dim=1).to(DEVICE)
        labels = ids.clone(); labels[:, 0] = -100
        with torch.no_grad(): loss = model(ids, labels=labels).loss.float()
        nlls.append(loss * chunk.shape[1]); n += chunk.shape[1]
    return round(torch.exp(torch.stack(nlls).sum() / n).item(), 2)

ppl = {}
ppl_run('fp16', lambda: None)
ppl_run('uniform_4bit', lambda: apply_neuron_bits(np.full((N_LAYERS, MLP_DIM), 4), 4))
ppl_run('none_mlp4_a8', lambda: apply_neuron_bits(np.full((N_LAYERS, MLP_DIM), 4), 8))
ppl_run('2tier_c4a8', lambda: apply_neuron_bits(tier_bits_arr(mlp_tiers, {0:16,1:4,2:4,3:4}), 8))
ppl_run('none_mlp3_a8', lambda: apply_neuron_bits(np.full((N_LAYERS, MLP_DIM), 3), 8))
ppl_run('2tier_c3a8', lambda: apply_neuron_bits(tier_bits_arr(mlp_tiers, {0:16,1:3,2:3,3:3}), 8))
json.dump(ppl, open(f'{OUT}/E4_wikitext_ppl.json','w'), indent=2)

fp16                         PPL=32.31
uniform_4bit                 PPL=36.52
none_mlp4_a8                 PPL=33.84
2tier_c4a8                   PPL=33.81
none_mlp3_a8                 PPL=47.43
2tier_c3a8                   PPL=47.58


In [19]:
cal_samples = []
for cs in ['CS1','CS2','CS3','CS4','CS5']:
    cal_samples.extend(json.load(open(f'/workspace/data/analysis_{cs}.json'))[:26])
cal_samples = cal_samples[:128]
hessians = {l: {} for l in range(N_LAYERS)}
_layer_inputs = {}
def make_hook(l, name):
    def hook_fn(module, inp, out): _layer_inputs.setdefault(f'{l}_{name}', []).append(inp[0].detach())
    return hook_fn
hooks = []
for l in range(N_LAYERS):
    layer = model.model.layers[l]
    for name in ['gate','up','down']: hooks.append(getattr(layer.mlp, f'{name}_proj').register_forward_hook(make_hook(l, name)))
    for name in ['q','k','v','o']:     hooks.append(getattr(layer.self_attn, f'{name}_proj').register_forward_hook(make_hook(l, name)))
restore(); t0 = time.time()
for i in tqdm(range(len(cal_samples)), desc='Hessians'):
    s = cal_samples[i]
    inputs = tokenizer(s['prompt'] + s['completion'], return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
    _layer_inputs.clear()
    with torch.no_grad(): model(**inputs)
    for key, xs in _layer_inputs.items():
        l, name = key.split('_', 1); l = int(l)
        for x in xs:
            x2 = x.reshape(-1, x.shape[-1]).float(); H = (x2.T @ x2).cpu()
            hessians[l][name] = H if name not in hessians[l] else hessians[l][name] + H
    _layer_inputs.clear()
for h in hooks: h.remove()
gc.collect(); torch.cuda.empty_cache()
print(f'Hessians done in {time.time()-t0:.0f}s; layer0 down H shape {tuple(hessians[0]["down"].shape)}')

Hessians:   0%|          | 0/128 [00:00<?, ?it/s]

Hessians done in 1099s; layer0 down H shape (9216, 9216)


In [20]:
from scipy import stats

def gptq_quantize_layer(W, H, bits_per_col=None, default_bits=4, block_size=128, damp=0.01, group_size=128):
    out_features, in_features = W.shape
    W = W.float().clone(); Q = torch.zeros_like(W)
    if bits_per_col is None: bits_per_col = [default_bits] * in_features
    H = H.float().clone(); H.diagonal().add_(damp * H.diag().mean())
    try: H_inv = torch.cholesky_inverse(torch.linalg.cholesky(H))
    except RuntimeError:
        H.diagonal().add_(0.1 * H.diag().mean())
        try: H_inv = torch.cholesky_inverse(torch.linalg.cholesky(H))
        except RuntimeError: H_inv = torch.linalg.pinv(H)
    for bs in range(0, in_features, block_size):
        be = min(bs + block_size, in_features); bl = be - bs
        Wb = W[:, bs:be].clone(); Hb = H_inv[bs:be, bs:be]; Err = torch.zeros(out_features, bl, device=W.device)
        for j in range(bl):
            w = Wb[:, j]; d = Hb[j, j].clamp(min=1e-10); bits = int(bits_per_col[bs + j])
            if bits >= 16: Q[:, bs + j] = w; continue
            if bits == 0: q = torch.zeros_like(w)
            else:
                n = w.numel(); pl = ((n + group_size - 1) // group_size) * group_size
                p = torch.zeros(pl, device=w.device); p[:n] = w; g = p.reshape(-1, group_size)
                gmin = g.min(1, keepdim=True).values; gmax = g.max(1, keepdim=True).values
                sc = ((gmax - gmin) / (2**bits - 1)).clamp(min=1e-10)
                q = (torch.round((g - gmin) / sc) * sc + gmin).reshape(-1)[:n]
            Q[:, bs + j] = q; err = (w - q) / d; Err[:, j] = err
            if j < bl - 1: Wb[:, j+1:] -= err.unsqueeze(1) * Hb[j, j+1:].unsqueeze(0)
        if be < in_features: W[:, be:] -= Err @ H_inv[bs:be, be:]
    return Q.to(torch.bfloat16)

def apply_gptq_paper(mlp_tiers, mlp_tier_bits, attn_bits):
    """Cell D verbatim: gate/up GPTQ'd as gathered row sub-matrices per tier; down with per-column bits; attn uniform."""
    for l in tqdm(range(N_LAYERS), desc='GPTQ paper', leave=False):
        layer = model.model.layers[l]
        for name in ['gate','up']:
            W0 = original_weights[l][name]; H = hessians[l][name].to(DEVICE); Qr = W0.clone()
            for tv in [1, 2, 3]:
                bits = mlp_tier_bits[tv]; idx = np.where(mlp_tiers[l] == tv)[0]
                if len(idx) == 0: continue
                it = torch.tensor(idx, device=DEVICE, dtype=torch.long)
                Qr[it, :] = 0 if bits == 0 else gptq_quantize_layer(W0[it, :], H, default_bits=bits)
            getattr(layer.mlp, f'{name}_proj').weight.data = Qr
        H = hessians[l]['down'].to(DEVICE)
        col_bits = [mlp_tier_bits[mlp_tiers[l][n]] for n in range(MLP_DIM)]
        layer.mlp.down_proj.weight.data = gptq_quantize_layer(original_weights[l]['down'], H, bits_per_col=col_bits)
        for name in ['q','k','v','o']:
            getattr(layer.self_attn, f'{name}_proj').weight.data = gptq_quantize_layer(
                original_weights[l][name], hessians[l][name].to(DEVICE), default_bits=attn_bits)
        torch.cuda.empty_cache()

def apply_gptq_aligned(mask, comp_bits, attn_bits):
    """GPTQ full MLP matrices at comp_bits, attn at attn_bits, then restore protected neurons from original."""
    for l in tqdm(range(N_LAYERS), desc='GPTQ aligned', leave=False):
        layer = model.model.layers[l]; keep = torch.tensor(np.where(mask[l])[0], device=DEVICE, dtype=torch.long)
        for name in ['gate','up','down']:
            Q = gptq_quantize_layer(original_weights[l][name], hessians[l][name].to(DEVICE), default_bits=comp_bits)
            if len(keep):
                if name in ['gate','up']: Q[keep, :] = original_weights[l][name][keep, :]
                else:                     Q[:, keep] = original_weights[l][name][:, keep]
            getattr(layer.mlp, f'{name}_proj').weight.data = Q
        for name in ['q','k','v','o']:
            getattr(layer.self_attn, f'{name}_proj').weight.data = gptq_quantize_layer(
                original_weights[l][name], hessians[l][name].to(DEVICE), default_bits=attn_bits)
        torch.cuda.empty_cache()

def perturbation_stats(tag):
    """Mean |W_now - W_orig| per neuron (gate row + up row + down col), skeleton vs non-skeleton; plus down-only."""
    sk, nsk, sk_d, nsk_d = [], [], [], []
    for l in range(N_LAYERS):
        layer = model.model.layers[l]
        dg = (layer.mlp.gate_proj.weight.data.float() - original_weights[l]['gate'].float()).abs().mean(1)
        du = (layer.mlp.up_proj.weight.data.float()   - original_weights[l]['up'].float()).abs().mean(1)
        dd = (layer.mlp.down_proj.weight.data.float() - original_weights[l]['down'].float()).abs().mean(0)
        per = ((dg + du + dd) / 3).cpu().numpy(); ddn = dd.cpu().numpy(); m = (mlp_tiers[l] == 0)
        sk.append(per[m]); nsk.append(per[~m]); sk_d.append(ddn[m]); nsk_d.append(ddn[~m])
    sk, nsk, sk_d, nsk_d = map(np.concatenate, (sk, nsk, sk_d, nsk_d))
    r = {'skeleton_all3': float(sk.mean()), 'nonskel_all3': float(nsk.mean()),
         'skeleton_down': float(sk_d.mean()), 'nonskel_down': float(nsk_d.mean()),
         'p_all3': float(stats.ttest_ind(sk, nsk, equal_var=False).pvalue),
         'p_down': float(stats.ttest_ind(sk_d, nsk_d, equal_var=False).pvalue)}
    print(f'   |dW| {tag}: all3 skel={r["skeleton_all3"]:.5f} non={r["nonskel_all3"]:.5f} | down-only skel={r["skeleton_down"]:.5f} non={r["nonskel_down"]:.5f}')
    return r

def run_gptq(name, apply_fn, measure=False, **meta):
    restore(); t0 = time.time(); apply_fn(); qt = round(time.time()-t0)
    pert = perturbation_stats(name) if measure else None
    r = evaluate_retention(); r.update(meta); r['quant_s'] = qt
    if pert: r['perturbation'] = pert
    restore(); json.dump(r, open(f'{OUT}/{name}.json','w'), indent=2)
    print(f"{name:32s} {r['retention']:6.2f}%  ({r['retained']}/{r['total']})  [quant {qt}s]"); return r

all_comp = np.full((N_LAYERS, MLP_DIM), 2, dtype=np.int8)
run_gptq('E6_gptq_uniform4',        lambda: apply_gptq_paper(all_comp, {0:16,1:8,2:4,3:0}, 4), config='paper GPTQ uniform 4 (reproduce 79.0)')
run_gptq('E6_gptq_paper_c4a8',      lambda: apply_gptq_paper(mlp_tiers, tier_bits, 8), measure=True, config='paper sub-matrix 4-tier c4+a8 (reproduce 89.5)')
run_gptq('E6_gptq_aligned_c3a8',    lambda: apply_gptq_aligned(mlp_tiers == 0, 3, 8), config='aligned GPTQ MLP3 + attn8, skeleton restored')
run_gptq('E6_gptq_aligned_none_c3a8', lambda: apply_gptq_aligned(np.zeros((N_LAYERS, MLP_DIM), bool), 3, 8), config='aligned GPTQ MLP3 + attn8, nothing protected')

GPTQ paper:   0%|          | 0/26 [00:00<?, ?it/s]

E6_gptq_uniform4                  96.19%  (101/105)  [quant 146s]


GPTQ paper:   0%|          | 0/26 [00:00<?, ?it/s]

   |dW| E6_gptq_paper_c4a8: all3 skel=0.00010 non=0.00078 | down-only skel=0.00029 non=0.00074
E6_gptq_paper_c4a8                98.10%  (103/105)  [quant 151s]


GPTQ aligned:   0%|          | 0/26 [00:00<?, ?it/s]

E6_gptq_aligned_c3a8              81.90%  (86/105)  [quant 139s]


GPTQ aligned:   0%|          | 0/26 [00:00<?, ?it/s]

E6_gptq_aligned_none_c3a8         81.90%  (86/105)  [quant 139s]


{'retention': 81.9,
 'retained': 86,
 'total': 105,
 'per_cs': {'CS1': {'retained': 28, 'total': 28},
  'CS2': {'retained': 20, 'total': 25},
  'CS3': {'retained': 19, 'total': 23},
  'CS4': {'retained': 10, 'total': 15},
  'CS5': {'retained': 9, 'total': 14}},
 'config': 'aligned GPTQ MLP3 + attn8, nothing protected',
 'quant_s': 139}

In [21]:
run_gptq('E6_gptq_uniform4_repeat', lambda: apply_gptq_paper(all_comp, {0:16,1:8,2:4,3:0}, 4), config='repeat — determinism check')
_orig = gptq_quantize_layer
def gptq_quantize_layer(W, H, **kw): return _orig(W, H, damp=0.1, **{k:v for k,v in kw.items() if k!='damp'})
run_gptq('E6_gptq_uniform4_damp0.1', lambda: apply_gptq_paper(all_comp, {0:16,1:8,2:4,3:0}, 4), config='damp=0.1 — sensitivity check')
gptq_quantize_layer = _orig
run_gptq('E6_gptq_uniform3',        lambda: apply_gptq_paper(all_comp, {0:16,1:8,2:3,3:0}, 3), config='paper GPTQ uniform 3 (paper had 81.9, naive 87.6)')

GPTQ paper:   0%|          | 0/26 [00:00<?, ?it/s]

E6_gptq_uniform4_repeat           96.19%  (101/105)  [quant 142s]


GPTQ paper:   0%|          | 0/26 [00:00<?, ?it/s]

E6_gptq_uniform4_damp0.1          88.57%  (93/105)  [quant 140s]


GPTQ paper:   0%|          | 0/26 [00:00<?, ?it/s]

E6_gptq_uniform3                  77.14%  (81/105)  [quant 143s]


{'retention': 77.14,
 'retained': 81,
 'total': 105,
 'per_cs': {'CS1': {'retained': 28, 'total': 28},
  'CS2': {'retained': 16, 'total': 25},
  'CS3': {'retained': 17, 'total': 23},
  'CS4': {'retained': 11, 'total': 15},
  'CS5': {'retained': 9, 'total': 14}},
 'config': 'paper GPTQ uniform 3 (paper had 81.9, naive 87.6)',
 'quant_s': 143}

In [22]:
run('E8_none_c3a4',     lambda: apply_protect(np.zeros((N_LAYERS, MLP_DIM), bool), 3, 4), config='aligned none, MLP3, attn4')
run('E8_skeleton_c3a4', lambda: apply_protect(mlp_tiers == 0, 3, 4), config='aligned tier skeleton, MLP3, attn4')
vals = [run(f'E8_random{s}_c3a4', lambda: apply_protect(rand_mask(s), 3, 4), config='aligned random', seed=s)['retention'] for s in [1,2,3]]
print(f"c3a4: none={_} skeleton={_} random={np.mean(vals):.1f}±{np.std(vals,ddof=1):.1f}")

E8_none_c3a4                      95.24%  (100/105)  [172s]
E8_skeleton_c3a4                  95.24%  (100/105)  [170s]
E8_random1_c3a4                   95.24%  (100/105)  [165s]
E8_random2_c3a4                   94.29%  (99/105)  [171s]
E8_random3_c3a4                   95.24%  (100/105)  [170s]
c3a4: none={'retention': 77.14, 'retained': 81, 'total': 105, 'per_cs': {'CS1': {'retained': 28, 'total': 28}, 'CS2': {'retained': 16, 'total': 25}, 'CS3': {'retained': 17, 'total': 23}, 'CS4': {'retained': 11, 'total': 15}, 'CS5': {'retained': 9, 'total': 14}}, 'config': 'paper GPTQ uniform 3 (paper had 81.9, naive 87.6)', 'quant_s': 143} skeleton={'retention': 77.14, 'retained': 81, 'total': 105, 'per_cs': {'CS1': {'retained': 28, 'total': 28}, 'CS2': {'retained': 16, 'total': 25}, 'CS3': {'retained': 17, 'total': 23}, 'CS4': {'retained': 11, 'total': 15}, 'CS5': {'retained': 9, 'total': 14}}, 'config': 'paper GPTQ uniform 3 (paper had 81.9, naive 87.6)', 'quant_s': 143} random=94.9±0.5


In [23]:
del model, original_weights, hessians; gc.collect(); torch.cuda.empty_cache()
print(f'VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB')
from huggingface_hub import snapshot_download
snapshot_download(repo_id='primal-sage/circuittier-llama8b-cypher', repo_type='model', local_dir='/workspace/llama',
                  allow_patterns=['models/finetuned/**', 'data/**', 'results/tier_maps.npz', 'results/tier_summary.json'])
import subprocess; print(subprocess.run('du -sh /workspace/llama/*; ls /workspace/llama/data /workspace/llama/models/finetuned', shell=True, capture_output=True, text=True).stdout)


VRAM after cleanup: 0.2 GB


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

base_model_accuracy.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

finetuned_accuracy.json: 0.00B [00:00, ?B/s]

analysis_500.json: 0.00B [00:00, ?B/s]

...

models/finetuned/final/tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

trainer_state.json: 0.00B [00:00, ?B/s]

results/tier_maps.npz:   0%|          | 0.00/209k [00:00<?, ?B/s]

tier_summary.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

94M	/workspace/llama/data
15G	/workspace/llama/models
1.2M	/workspace/llama/results
/workspace/llama/data:
analysis_500.json
base_model_accuracy.json
finetuned_accuracy.json
test_CY1.json
test_CY2.json
test_CY3.json
test_CY4.json
test_CY5.json
test_processed.json
train_processed.json

/workspace/llama/models/finetuned:
final



In [25]:
from transformers import PreTrainedTokenizerFast
tokenizer = PreTrainedTokenizerFast(tokenizer_file=f'{MP}/tokenizer.json',
                                    bos_token='<|begin_of_text|>', eos_token='<|end_of_text|>', pad_token='<|reserved_special_token_0|>')
tokenizer.padding_side = 'left'
print('bos', tokenizer.bos_token_id, 'eos', tokenizer.eos_token_id, 'pad', tokenizer.pad_token_id, '(expect 128000/128001/128002)')
print('adds BOS:', tokenizer('hi').input_ids[:2], '| VRAM', round(torch.cuda.memory_allocated()/1e9,1), 'GB')
NL, MLP_L = model.config.num_hidden_layers, model.config.intermediate_size; print('layers', NL, 'mlp', MLP_L)

test_l = json.load(open(f'{LP}/data/test_processed.json'))
accp = [p for p in [f'{LP}/data/finetuned_accuracy.json', f'{LP}/data/test_results.json'] if os.path.exists(p)]
print('accuracy file:', accp, '| data dir:', os.listdir(f'{LP}/data'))
acc = json.load(open(accp[0])); bcl = acc['baseline_correct']
ordered = sorted(i for cy in bcl for i in bcl[cy])[:1000]
sub = [test_l[i] for i in ordered]
from collections import Counter; print('1k split per CY:', dict(sorted(Counter(s['complexity'] for s in sub).items())), '(expect 76/356/219/128/221)')

tm = np.load(f'{LP}/results/tier_maps.npz', allow_pickle=True)
for k in tm.keys():
    a = tm[k]; print(f'  {k}: shape={a.shape} dtype={a.dtype} uniq={np.unique(a)[:8]} counts={np.bincount(a.ravel().astype(int)) if a.dtype.kind in "iub" else ""}')

bos 128000 eos 128001 pad 128002 (expect 128000/128001/128002)
adds BOS: [128000, 6151] | VRAM 16.3 GB
layers 32 mlp 14336
accuracy file: ['/workspace/llama/data/finetuned_accuracy.json'] | data dir: ['train_processed.json', 'test_processed.json', 'test_CY3.json', 'test_CY4.json', 'test_CY2.json', 'test_CY5.json', 'test_CY1.json', 'analysis_500.json', 'finetuned_accuracy.json', 'base_model_accuracy.json']
1k split per CY: {'CY1': 69, 'CY2': 366, 'CY3': 213, 'CY4': 123, 'CY5': 229} (expect 76/356/219/128/221)
  mlp_votes: shape=(32, 14336) dtype=int32 uniq=[0 1 2 3 4 5 6] counts=[378141  47563  16914   9512   5454   1123     45]
  mlp_skeleton: shape=(32, 14336) dtype=bool uniq=[False  True] counts=[442618  16134]
  mlp_supporting: shape=(32, 14336) dtype=bool uniq=[False  True] counts=[394275  64477]
  mlp_compressible: shape=(32, 14336) dtype=bool uniq=[False  True] counts=[ 80611 378141]
  attn_votes: shape=(32, 32) dtype=int32 uniq=[0 1 2 3 4] counts=[805 142  62  14   1]
  attn_ske

In [26]:
snapshot_download(repo_id='primal-sage/circuittier-llama8b-cypher', repo_type='model', local_dir=LP,
                  allow_patterns=['results/ALL_SIGNALS_NORMALIZED.npz'])
sn = np.load(f'{LP}/results/ALL_SIGNALS_NORMALIZED.npz', allow_pickle=True)
mkeys = [k for k in sn.keys() if sn[k].shape == (NL, MLP_L)]; print('signal keys:', mkeys)
S_l = np.stack([sn[k] for k in mkeys])
def llama_skeleton(P):
    thr = np.percentile(S_l.reshape(len(mkeys), -1), P, axis=1)[:, None, None]
    return (S_l >= thr).sum(0) >= 3
print('P=95 rebuild matches saved map:', np.array_equal(llama_skeleton(95), tm['mlp_skeleton']))
skel99 = llama_skeleton(99); print('P=99 skeleton:', int(skel99.sum()), '(expect 2133)')

orig_l = {l: {n: getattr(model.model.layers[l].mlp if n in ['gate','up','down'] else model.model.layers[l].self_attn, f'{n}_proj').weight.data.to('cpu', copy=True)
              for n in ['gate','up','down','q','k','v','o']} for l in range(NL)}
def restore_l():
    for l in range(NL):
        for n, W in orig_l[l].items():
            getattr(model.model.layers[l].mlp if n in ['gate','up','down'] else model.model.layers[l].self_attn, f'{n}_proj').weight.data.copy_(W.to(DEVICE))
def apply_protect_l(mask, mlp_bits, attn_bits):
    for l in range(NL):
        layer = model.model.layers[l]; keep = torch.tensor(np.where(mask[l])[0], device=DEVICE, dtype=torch.long)
        for n in ['gate','up','down']:
            W0 = orig_l[l][n].to(DEVICE); Q = naive_quantize(W0, mlp_bits)
            if len(keep):
                if n in ['gate','up']: Q[keep, :] = W0[keep, :]
                else:                  Q[:, keep] = W0[:, keep]
            getattr(layer.mlp, f'{n}_proj').weight.data = Q
        for n in ['q','k','v','o']:
            getattr(layer.self_attn, f'{n}_proj').weight.data = naive_quantize(orig_l[l][n].to(DEVICE), attn_bits)
        torch.cuda.empty_cache()

class StopAllNewline(StoppingCriteria):
    def __init__(self, L): self.L = L
    def __call__(self, ids, scores, **kw):
        return torch.tensor([('\n' in tokenizer.decode(r[self.L:], skip_special_tokens=True)) or (tokenizer.eos_token_id in r[self.L:].tolist())
                             for r in ids], device=ids.device)
def eval_llama(bs=8):
    retained = 0; per = {}
    for b in tqdm(range(0, len(sub), bs), leave=False):
        batch = sub[b:b+bs]
        enc = tokenizer([s['prompt'] for s in batch], return_tensors='pt', padding=True, truncation=True, max_length=480).to(DEVICE)
        L = enc['input_ids'].shape[1]
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.pad_token_id,
                                 stopping_criteria=StoppingCriteriaList([StopAllNewline(L)]))
        for s, row in zip(batch, out):
            gen = tokenizer.decode(row[L:], skip_special_tokens=True).strip().split('\n')[0].strip()
            ok = gen == s['cypher_ref'].strip(); retained += ok
            d = per.setdefault(s['complexity'], {'retained':0,'total':0}); d['total'] += 1; d['retained'] += ok
    return {'retention': round(100*retained/len(sub), 2), 'retained': retained, 'total': len(sub), 'per_cy': per}
def run_l(name, apply_fn, **meta):
    restore_l(); apply_fn(); t0 = time.time(); r = eval_llama(); r.update(meta); r['eval_s'] = round(time.time()-t0); restore_l()
    json.dump(r, open(f'{OUT}/{name}.json','w'), indent=2); print(f"{name:32s} {r['retention']:6.2f}%  ({r['retained']}/{r['total']})  [{r['eval_s']}s]"); return r
print('ready; VRAM', round(torch.cuda.memory_allocated()/1e9,1), 'GB')

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

results/ALL_SIGNALS_NORMALIZED.npz:   0%|          | 0.00/9.55M [00:00<?, ?B/s]

signal keys: ['S1_eap_mlp', 'S2_gradient_mlp', 'S3_magnitude_mlp', 'S4_weight_delta_mlp', 'S5_act_delta_mlp', 'S6_edge_mlp']
P=95 rebuild matches saved map: True
P=99 skeleton: 2133 (expect 2133)
ready; VRAM 16.3 GB


In [28]:
model.generation_config.temperature = None; model.generation_config.top_p = None   # silence the sampling warnings
def eval_llama(bs=8):
    retained = 0; per = {}
    for b in tqdm(range(0, len(sub), bs), leave=False):
        batch = sub[b:b+bs]
        enc = tokenizer([s['prompt'] for s in batch], return_tensors='pt', padding=True, truncation=True, max_length=480)
        ids, am = enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE); L = ids.shape[1]
        with torch.no_grad():
            out = model.generate(input_ids=ids, attention_mask=am, max_new_tokens=150, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id, stopping_criteria=StoppingCriteriaList([StopAllNewline(L)]))
        for s, row in zip(batch, out):
            gen = tokenizer.decode(row[L:], skip_special_tokens=True).strip().split('\n')[0].strip()
            ok = gen == s['cypher_ref'].strip(); retained += ok
            d = per.setdefault(s['complexity'], {'retained':0,'total':0}); d['total'] += 1; d['retained'] += ok
    return {'retention': round(100*retained/len(sub), 2), 'retained': retained, 'total': len(sub), 'per_cy': per}

In [29]:
run_l('E9_llama_uniform4',   lambda: apply_protect_l(np.zeros((NL, MLP_L), bool), 4, 4), config='no skeleton = uniform 4-bit (theirs 93.4)')
run_l('E9_llama_skelP99',    lambda: apply_protect_l(skel99, 4, 4), config='P=99 skeleton @16 (2133), rest @4, attn @4 (theirs c4a4_s4 97.1)')
vals = []
for s in [1, 2, 3]:
    rng = np.random.RandomState(s); m = np.zeros(NL*MLP_L, bool); m[rng.choice(NL*MLP_L, int(skel99.sum()), replace=False)] = True
    vals.append(run_l(f'E9_llama_random{s}', lambda: apply_protect_l(m.reshape(NL, MLP_L), 4, 4), config='random 2133 @16', seed=s)['retention'])
print(f'\nLLAMA c4a4: random={np.mean(vals):.1f}±{np.std(vals,ddof=1):.1f}')

  0%|          | 0/125 [00:00<?, ?it/s]

E9_llama_uniform4                 71.10%  (711/1000)  [285s]


  0%|          | 0/125 [00:00<?, ?it/s]

E9_llama_skelP99                  82.70%  (827/1000)  [286s]


  0%|          | 0/125 [00:00<?, ?it/s]

E9_llama_random1                  71.00%  (710/1000)  [284s]


  0%|          | 0/125 [00:00<?, ?it/s]

E9_llama_random2                  71.00%  (710/1000)  [284s]


  0%|          | 0/125 [00:00<?, ?it/s]

E9_llama_random3                  70.80%  (708/1000)  [284s]

LLAMA c4a4: random=70.9±0.1


In [30]:
# (a) fp16 through OUR batched eval: these are baseline-correct samples, so this must be ~100%
fp = run_l('E9_llama_fp16', lambda: None, config='no quantization, batched eval')

# (b) uniform 4-bit, SEQUENTIAL (unbatched) on the first 100 samples — isolates batching
restore_l(); apply_protect_l(np.zeros((NL, MLP_L), bool), 4, 4)
seq_ok = 0
for s in tqdm(sub[:100], leave=False):
    enc = tokenizer(s['prompt'], return_tensors='pt', truncation=True, max_length=480)
    ids = enc['input_ids'].to(DEVICE); L = ids.shape[1]
    with torch.no_grad():
        out = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=150, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id, stopping_criteria=StoppingCriteriaList([StopOnNewline(L)]))
    seq_ok += tokenizer.decode(out[0][L:], skip_special_tokens=True).strip().split('\n')[0].strip() == s['cypher_ref'].strip()
restore_l()
# batched result on the same 100 for comparison
r100 = json.load(open(f'{OUT}/E9_llama_uniform4.json'))
print(f'uniform4 sequential first-100: {seq_ok}/100   (batched full run was {r100["retention"]}%)')

  0%|          | 0/125 [00:00<?, ?it/s]

E9_llama_fp16                     89.40%  (894/1000)  [287s]


  0%|          | 0/100 [00:00<?, ?it/s]

uniform4 sequential first-100: 64/100   (batched full run was 71.1%)


In [31]:
p0 = sub[0]['prompt']; print('prompt starts:', repr(p0[:60])); print('first ids:', tokenizer(p0).input_ids[:3], '(double 128000 = BOS added twice)')

def eval_llama_ps(bs=8):
    ok_list = []
    for b in tqdm(range(0, len(sub), bs), leave=False):
        batch = sub[b:b+bs]
        enc = tokenizer([s['prompt'] for s in batch], return_tensors='pt', padding=True, truncation=True, max_length=480)
        ids, am = enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE); L = ids.shape[1]
        with torch.no_grad():
            out = model.generate(input_ids=ids, attention_mask=am, max_new_tokens=150, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id, stopping_criteria=StoppingCriteriaList([StopAllNewline(L)]))
        for s, row in zip(batch, out):
            ok_list.append(tokenizer.decode(row[L:], skip_special_tokens=True).strip().split('\n')[0].strip() == s['cypher_ref'].strip())
    return np.array(ok_list)

PS = {}
def run_ps(name, apply_fn):
    restore_l(); apply_fn(); PS[name] = eval_llama_ps(); restore_l()
    np.save(f'{OUT}/{name}_persample.npy', PS[name]); print(f'{name:22s} raw {PS[name].mean()*100:.1f}%')
run_ps('fp16',     lambda: None)
run_ps('uniform4', lambda: apply_protect_l(np.zeros((NL, MLP_L), bool), 4, 4))
run_ps('skelP99',  lambda: apply_protect_l(skel99, 4, 4))
for s in [1, 2, 3]:
    rng = np.random.RandomState(s); m = np.zeros(NL*MLP_L, bool); m[rng.choice(NL*MLP_L, int(skel99.sum()), replace=False)] = True
    run_ps(f'random{s}', lambda: apply_protect_l(m.reshape(NL, MLP_L), 4, 4))

base = PS['fp16']; n = int(base.sum()); cy = np.array([s['complexity'] for s in sub])
print(f'\nRe-based on {n} fp16-correct samples:')
rb = {}
for k in PS:
    if k == 'fp16': continue
    rb[k] = 100 * (PS[k] & base).sum() / n
    per = {c: f"{(PS[k] & base & (cy==c)).sum()}/{(base & (cy==c)).sum()}" for c in sorted(set(cy))}
    print(f'  {k:10s} {rb[k]:5.1f}%   {per}')
rv = [rb[f'random{s}'] for s in [1,2,3]]
print(f'\nLLAMA c4a4 re-based: none={rb["uniform4"]:.1f}  skeleton={rb["skelP99"]:.1f}  random={np.mean(rv):.1f}±{np.std(rv,ddof=1):.1f}')
json.dump({'n_base': n, 'rebased': rb, 'random_mean': np.mean(rv), 'random_std': np.std(rv, ddof=1)}, open(f'{OUT}/E9_llama_rebased_summary.json','w'), indent=2)

prompt starts: 'Schema: Graph schema: Relevant node labels and their propert'
first ids: [128000, 8802, 25] (double 128000 = BOS added twice)


  0%|          | 0/125 [00:00<?, ?it/s]

fp16                   raw 89.4%


  0%|          | 0/125 [00:00<?, ?it/s]

uniform4               raw 71.1%


...

random1                raw 71.0%


  0%|          | 0/125 [00:00<?, ?it/s]

random2                raw 71.0%


  0%|          | 0/125 [00:00<?, ?it/s]

random3                raw 70.8%

Re-based on 894 fp16-correct samples:
  uniform4    77.1%   {'CY1': '64/67', 'CY2': '293/326', 'CY3': '124/177', 'CY4': '66/113', 'CY5': '142/211'}
  skelP99     91.5%   {'CY1': '63/67', 'CY2': '314/326', 'CY3': '165/177', 'CY4': '93/113', 'CY5': '183/211'}
  random1     77.1%   {'CY1': '63/67', 'CY2': '292/326', 'CY3': '123/177', 'CY4': '67/113', 'CY5': '144/211'}
  random2     77.1%   {'CY1': '63/67', 'CY2': '295/326', 'CY3': '123/177', 'CY4': '66/113', 'CY5': '142/211'}
  random3     76.8%   {'CY1': '64/67', 'CY2': '290/326', 'CY3': '124/177', 'CY4': '66/113', 'CY5': '143/211'}

LLAMA c4a4 re-based: none=77.1  skeleton=91.5  random=77.0±0.1


In [32]:
K = int(skel99.sum()); names = ['eap','gradient','magnitude','weight_delta','act_delta','edge']
for i, nm in enumerate(names):
    m = np.zeros(NL*MLP_L, bool); m[np.argpartition(S_l[i].ravel(), -K)[-K:]] = True
    run_ps(f'single_{nm}', lambda: apply_protect_l(m.reshape(NL, MLP_L), 4, 4))
print(f'\nRe-based (n={n}):  6-signal consensus = {rb["skelP99"]:.1f}')
for nm in names:
    v = 100 * (PS[f'single_{nm}'] & base).sum() / n
    ov = int((PS and np.zeros(1)).sum())  # placeholder to keep line simple
    m = np.zeros(NL*MLP_L, bool); m[np.argpartition(S_l[names.index(nm)].ravel(), -K)[-K:]] = True
    print(f'  {nm:13s} {v:5.1f}%   overlap with consensus skeleton: {int((m.reshape(NL,MLP_L) & skel99).sum())}/{K}')
    rb[f'single_{nm}'] = v
json.dump({'n_base': n, 'rebased': rb}, open(f'{OUT}/E9_llama_rebased_summary.json','w'), indent=2)

  0%|          | 0/125 [00:00<?, ?it/s]

single_eap             raw 81.7%


  0%|          | 0/125 [00:00<?, ?it/s]

single_gradient        raw 82.8%


  0%|          | 0/125 [00:00<?, ?it/s]

...

single_weight_delta    raw 71.1%


  0%|          | 0/125 [00:00<?, ?it/s]

single_act_delta       raw 71.4%


  0%|          | 0/125 [00:00<?, ?it/s]

single_edge            raw 70.8%

Re-based (n=894):  6-signal consensus = 91.5
  eap            90.6%   overlap with consensus skeleton: 1028/2133
  gradient       91.9%   overlap with consensus skeleton: 623/2133
  magnitude      91.4%   overlap with consensus skeleton: 674/2133
  weight_delta   77.1%   overlap with consensus skeleton: 82/2133
  act_delta      77.5%   overlap with consensus skeleton: 994/2133
  edge           76.8%   overlap with consensus skeleton: 933/2133
